## Topic: Json Schema

### 1. Introduction of Json Schema

- Definition:
    - JSON Schema is a standard way to describe and validate the expected structure, data types, and constraints of JSON data.

    - JSON Schema → standardized JSON data contract


### Key Pros of JSON Schema


- 1. Language-independent — 
    - Can be used across Python, JavaScript, Java, Go, etc.

- 2. Standardized — 
    - Provides a widely recognized way to describe JSON data.

- 3. Explicit structure — 
    - Clearly defines properties, types, and required fields.

- 4. Supports constraints — 
    - Can define minimum, maximum, string length, array rules, etc.

- 5. Excellent for APIs and data exchange — 
    - Different systems can agree on the same data contract.

- 6. Useful for LLM structured output — 
    - Can describe the expected format of model responses.

### Key Cons of JSON Schema

- More verbose than Pydantic for Python developers.

- Less Pythonic because the schema is represented as JSON/dictionaries.

- Can become complex for deeply nested or highly constrained data.

- Less convenient for Python-specific validation logic compared with Pydantic.

- Requires understanding JSON Schema syntax such as type, properties, required, items, and constraints.

### Final Key Takeaway

- JSON Schema is a standardized way to define the expected structure, data types, required fields, and constraints of JSON data. In GenAI applications, it provides a clear contract between the LLM and downstream software systems.

### When to use what?
- 1. Use TypedDict (if):
    - we only need type hints.
    - we don't need validation (eg. checking numbers are positive).
    - we trust the LLM to return correct data.

- 2. Use Pydantic (if):
    - we need data validation (eg. sentiment must be "positive", "neutral", "negative").

    - we need default values if the LLM misses fields.

    - we want automatic type conversion (eg. "100" - 100).

- 3. Use Json Schema (if):
    - we don't want to import extra python library (pydantic)
    - we need validation but don't need python objects.
    - we want to define structure in a standard JSON format.

In [ ]:
"""  
STRUCTURED OUTPUT
│
├── TypedDict
│     └── Dictionary structure + type hints
│
├── Pydantic
│     └── Python model + validation
│
└── JSON Schema
      └── Standard JSON structure + constraints

"""

## A few things to remember with_structured_output() method

- inside the with_structured_output(
    - two method
    - 1. json mode
    - 2. function calling
)

- when use 1. json mode method:
    - when we need the structured output into json format.
    - eg: use model: gemini, claude

-  2. function calling use:
    - whe we need to the structured output into calling function
    - eg: Building Agents --> calling tools
    - eg: use model : GPT model 


- Key note:
    - some of open-source model can't support json and function calling method to get the structure output.
    - for these case, we manually doing the structure output format. 

In [ ]:
# example of open source model

from dotenv import load_dotenv
from typing import Optional, Literal
from pydantic import BaseModel, Field
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

# schema
class Review(BaseModel):

    key_themes: list[str] = Field(description="Write down all the key themes discussed in the review in a list")
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["pos", "neg"] = Field(description="Return sentiment of the review either negative, positive or neutral")
    pros: Optional[list[str]] = Field(default=None, description="Write down all the pros inside a list")
    cons: Optional[list[str]] = Field(default=None, description="Write down all the cons inside a list")
    name: Optional[str] = Field(default=None, description="Write the name of the reviewer")
    

structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by KzRaihan
""")

print(result)



- This code raise error because the TinyLlama/TinyLlama-1.1B-Chat-v1.0 open-source model can't support the structure output format.